# Polarized light and geometric phase in qg units

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Viny2030/qang/blob/main/notebooks/qang_optics.ipynb)

A polarization state of light is a qubit: the Poincaré sphere is the Bloch sphere. This notebook uses the published library (`pip install qang`, modules `qang.polarization` and `qang.geometric`) to show that the normalized **Stokes parameters are qg values**, that a lossless **Mueller matrix is the qg gate rule**, **Malus's law** in qg units, polarimetry from photon counts, and the **geometric phase** as Stokes' theorem on the sphere. Each result is computed twice, with qang and with standard optics or quantum mechanics, and the two are compared.

Note: `manuscript/qang_optics.pdf` (RESEARCH_NOTES §83–§84). The physics is standard; what is new is writing it in qang's units and checking it numerically.

## 1. Install

In [ ]:
!pip install -q -U "qang>=0.6.0" matplotlib
import qang
print('qang', qang.__version__)

## 2. Stokes parameters are qg values

With |H⟩ = |0⟩ and |V⟩ = |1⟩: qg_Z = S₁/S₀ (H − V), qg_X = S₂/S₀ (D − A), qg_Y = S₃/S₀ (R − L), with R = (|H⟩ + i|V⟩)/√2. The qg values from `qang.polarization` are compared with ⟨P⟩ computed by `qang.formulation` from the state.

In [ ]:
LABELS = ('state', 'with qang', 'standard', 'difference')
PLOT_MALUS = ('polarizer angle θ (rad)', 'transmitted intensity I/I₀')
PLOT_PHASE = ('qg_Z of the loop', 'geometric phase (turns)', '(qg_Z − 1)/2', 'overlaps', 'solid angle', 'curvature flux')
import numpy as np
import matplotlib.pyplot as plt
from qang import formulation as F
from qang import geometric as G
from qang import polarization as P

s2 = np.sqrt(2)
states = {"H": [1, 0], "V": [0, 1], "D": [1 / s2, 1 / s2], "A": [1 / s2, -1 / s2],
          "R": [1 / s2, 1j / s2], "L": [1 / s2, -1j / s2], "elliptic": [0.6, 0.8 * np.exp(0.7j)]}
print(f"{LABELS[0]:<10}{LABELS[1]:>34}{LABELS[2]:>34}{LABELS[3]:>12}")
for name, E in states.items():
    q = P.jones_to_qg(E)                                   # with qang: Stokes -> qg
    ref = F.qg_values(np.array(E, dtype=complex))          # standard: <P> from the state
    ref = {k: ref.get(k, 0.0) for k in "XYZ"}
    diff = max(abs(q[k] - ref[k]) for k in "XYZ")
    fmt = lambda d: "(" + ", ".join(f"{d[k]:+.3f}" for k in "XYZ") + ")"
    print(f"{name:<10}{fmt(q):>34}{fmt(ref):>34}{diff:>12.1e}")

## 3. Partially polarized light is a mixed state

The degree of polarization is the length of the qg vector; the purity is (1 + P²)/2.

In [ ]:
E = np.array([0.6, 0.8 * np.exp(0.7j)])
for p in (0.0, 0.3, 0.6, 0.9):
    S = P.depolarizer(1 - p) @ P.jones_to_stokes(E)
    q = P.stokes_to_qg(S)
    rho = F.state_from_qg({"I": 1.0, **q}, 1)
    print(f"depolarization {p:.1f}: P = {P.degree_of_polarization(S):.3f}, "
          f"purity (qang) {P.purity_from_stokes(S):.4f}, Tr rho^2 {np.real(np.trace(rho @ rho)):.4f}")

## 4. A lossless Mueller matrix is the qg gate rule

Mueller matrix (optics) against `qang.formulation.apply_gate` (qg'_P = qg_{U†PU}) for wave plates.

In [ ]:
rng = np.random.default_rng(1)
for name, J in {"half-wave 0.3": P.half_wave_plate(0.3), "quarter-wave 1.1": P.quarter_wave_plate(1.1),
                "rotator 0.7": P.rotator(0.7)}.items():
    err = 0.0
    for _ in range(5):
        E = rng.normal(size=2) + 1j * rng.normal(size=2); E /= np.linalg.norm(E)
        q_in = P.jones_to_qg(E)
        q_mueller = P.stokes_to_qg(P.mueller_from_jones(J) @ P.qg_to_stokes(q_in))   # optics
        q_rule = F.apply_gate({"I": 1.0, **q_in}, J)                                   # qang gate rule
        err = max(err, max(abs(q_mueller[k] - q_rule.get(k, 0.0)) for k in "XYZ"))
    print(f"{name:<18} max difference Mueller vs qg rule: {err:.1e}")

## 5. Malus's law in qg units

Behind a polarizer at angle θ: I = I₀ (1 + qg_Z cos 2θ + qg_X sin 2θ)/2, for any input. Lines: the qg formula. Points: the Mueller matrix of the polarizer.

In [ ]:
theta = np.linspace(0, np.pi, 200)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
for (name, E), c in zip([("H", [1, 0]), ("D", [1 / s2, 1 / s2]), ("R", [1 / s2, 1j / s2]),
                         ("elliptic", [0.6, 0.8 * np.exp(0.7j)])], ["#2a78d6", "#eb6834", "#1baf7a", "#5f5e58"]):
    q = P.jones_to_qg(E)
    ax.plot(theta, [P.malus_intensity(q, t) for t in theta], color=c, lw=2, label=name)
    pts = np.linspace(0, np.pi, 9)
    ax.plot(pts, [(P.mueller_from_jones(P.linear_polarizer(t)) @ P.qg_to_stokes(q))[0] for t in pts], "o", color=c, ms=5)
ax.set_xlabel(PLOT_MALUS[0]); ax.set_ylabel(PLOT_MALUS[1]); ax.legend(frameon=False, ncol=4, fontsize=8, loc="upper center")
ax.set_ylim(-0.05, 1.2); ax.grid(alpha=0.3)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.show()

## 6. Polarimetry from photon counts

Counts behind the six analyser settings give the three qg values with intervals (`qang.statistics.qg_estimate`).

In [ ]:
rng = np.random.default_rng(7)
E = np.array([0.6, 0.8 * np.exp(0.7j)]); q_true = P.jones_to_qg(E)
n = 1000
nH = rng.binomial(n, (1 + q_true["Z"]) / 2); nD = rng.binomial(n, (1 + q_true["X"]) / 2); nR = rng.binomial(n, (1 + q_true["Y"]) / 2)
est = P.qg_from_counts(nH, n - nH, nD, n - nD, nR, n - nR, method="wilson")
for k in "XYZ":
    iv = est["intervals"][k]
    print(f"qg_{k}: true {q_true[k]:+.3f}, estimate {est[k]:+.3f}, 95% interval [{iv.low:+.3f}, {iv.high:+.3f}]")
print(f"degree of polarization: estimate {est['P']:.3f} (true 1.000; shot noise can push it above 1)")

## 7. Geometric phase: Stokes' theorem on the sphere

A loop at constant qg_Z around the Z axis encloses Ω = 2π(1 − qg_Z), and the geometric phase is −Ω/2, i.e. **(qg_Z − 1)/2 of a turn**. Computed three ways: state overlaps (Pancharatnam), solid angle, and the flux of the Berry curvature (Stokes' theorem).

In [ ]:
def cone(qg_z, n):
    s = np.sqrt(1 - qg_z**2); t = np.linspace(0, 2 * np.pi, n, endpoint=False)
    return np.stack([s * np.cos(t), s * np.sin(t), np.full(n, qg_z)], axis=1)

zs = np.linspace(-0.9, 0.9, 7)
over = [G.geometric_phase_qg(cone(z, 2000)).phi for z in zs]                                   # overlaps
solid = [(G.berry_phase_from_solid_angle(G.solid_angle(cone(z, 2000), reference=[0, 0, 1])) / (2 * np.pi)) % 1 for z in zs]
flux = [(-G.curvature_flux(z) / (2 * np.pi)) % 1 for z in zs]                                 # Stokes: flux of the curvature
zz = np.linspace(-1, 0.995, 200)  # (qg_Z - 1)/2 mod 1 jumps to 0 at qg_Z = 1
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(zz, ((zz - 1) / 2) % 1, color="#5f5e58", lw=2, label=PLOT_PHASE[2])
ax.plot(zs, over, "o", color="#2a78d6", ms=8, label=PLOT_PHASE[3])
ax.plot(zs, solid, "s", color="#eb6834", ms=5, label=PLOT_PHASE[4])
ax.plot(zs, flux, "^", color="#1baf7a", ms=5, label=PLOT_PHASE[5])
ax.set_xlabel(PLOT_PHASE[0]); ax.set_ylabel(PLOT_PHASE[1]); ax.legend(frameon=False, fontsize=8); ax.grid(alpha=0.3)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.show()
print("max difference from (qg_Z - 1)/2:", max(abs(((a - (z - 1) / 2) + 0.5) % 1 - 0.5) for a, z in zip(over, zs)))

## 8. Pancharatnam's phase for polarized light

The loop H → D → R → H encloses an octant of the Poincaré sphere (Ω = π/2), so γ = −π/4.

In [ ]:
H, D, R = np.array([1, 0]), np.array([1, 1]) / s2, np.array([1, 1j]) / s2
path = [[P.jones_to_qg(E)[k] for k in "XYZ"] for E in (H, D, R)]
print("solid angle:", G.solid_angle(path), " (pi/2 =", np.pi / 2, ")")
print("Pancharatnam phase:", G.pancharatnam_phase([H, D, R]), " (-pi/4 =", -np.pi / 4, ")")

## Summary

* Stokes parameters = qg values; degree of polarization = length of the qg vector.
* Lossless Mueller matrix = the qg gate rule; Malus's law needs only qg_Z and qg_X.
* Geometric phase of a cone loop = (qg_Z − 1)/2 turns, checked three ways (Stokes' theorem).

Tests: `tests/test_polarization.py`, `tests/test_geometric.py` (26 checks).